In [1]:
# -------- Config --------
DATADIR = "/work/gr-fe/bryan/data/YEAST/"
COHORT = ""
META_PICKLE = f"{DATADIR}/{COHORT}/02_processed/phenotype.processed.pkl"
OMICS = ["EXPR"]  # datExpr_{omic}.csv for each
RAW_OMICS_DIR = f"{DATADIR}/{COHORT}/01_raw/"
OUT_DIR = f"{DATADIR}/{COHORT}/02_processed"
OVERWRITE = True  # overwrite existing output pickles

# -------- Imports --------
import os
import numpy as np
import pickle
from pathlib import Path
import pandas as pd

# -------- Helpers --------
def load_meta_ids(meta_path: str) -> pd.Series:
    """
    Load a pickle expected to contain either:
      - a pandas DataFrame with column 'ID', or
      - a dict with key 'ID' (array-like)
    Returns a clean string Series of IDs (duplicates removed, NaNs dropped).
    """
    # Try pandas first (faster for DataFrame pickles), fall back to pickle.load
    meta = None
    try:
        meta = pd.read_pickle(meta_path)
    except Exception:
        with open(meta_path, "rb") as f:
            meta = pickle.load(f)

    if isinstance(meta, pd.DataFrame) and "ID" in meta.columns:
        ids = meta["ID"]
    elif isinstance(meta, dict) and "ID" in meta:
        ids = pd.Series(meta["ID"], name="ID")
    else:
        raise ValueError("Meta pickle must be a DataFrame with 'ID' column or a dict with key 'ID'.")

    ids = ids.dropna().astype(str)
    ids = ids[~ids.duplicated()].reset_index(drop=True)
    return ids

def orient_to_ids(df: pd.DataFrame, id_set: set) -> pd.DataFrame | None:
    """
    Ensure sample IDs are in the index; if they're in columns, transpose.
    If neither index nor columns contain any meta IDs, return None.
    """
    # Normalize types to str for safe matching
    df.index = df.index.map(str)
    df.columns = df.columns.map(str)

    idx_hit = len(id_set.intersection(df.index))
    col_hit = len(id_set.intersection(df.columns))

    if idx_hit == 0 and col_hit > 0:
        df = df.T
        idx_hit = len(id_set.intersection(df.index))

    if idx_hit == 0:
        return None
    return df

# -------- Run --------
out_dir = Path(OUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

ids = load_meta_ids(META_PICKLE)
id_set = set(ids)

print(f"Loaded {len(ids)} unique meta IDs from: {META_PICKLE}")
print(f"Output dir: {out_dir}\n")

results = {}
for omic in OMICS:
    src = Path(RAW_OMICS_DIR) / f"datExpr_{omic}.csv"
    if not src.exists():
        print(f"[{omic}] SKIP - not found: {src}")
        continue

    try:
        df = pd.read_csv(src, index_col=0, dtype=str)
    except Exception as e:
        print(f"[{omic}] ERROR reading {src}: {e}")
        continue

    df_oriented = orient_to_ids(df, id_set)
    if df_oriented is None:
        print(f"[{omic}] SKIP - no overlap between meta IDs and {src.name} (rows or columns).")
        continue

    # Preserve meta ID order in the subset
    keep_ids = [i for i in ids if i in df_oriented.index]
    sub = df_oriented.loc[keep_ids].astype(np.float32) 

    out_path = out_dir / f"{omic}.pkl"
    if out_path.exists() and not OVERWRITE:
        print(f"[{omic}] Exists, not overwritten: {out_path} (shape={sub.shape})")
    else:
        with open(out_path, "wb") as f:
            pickle.dump({"expr": sub}, f, protocol=pickle.HIGHEST_PROTOCOL)
        print(f"[{omic}] Saved {sub.shape} to {out_path}")

    results[omic] = sub

# Optional: quick peek at one modality
for omic in OMICS:
    if omic in results:
        display(results[omic].head())
        break


Loaded 2417 unique meta IDs from: /work/gr-fe/bryan/data/YEAST///02_processed/phenotype.processed.pkl
Output dir: /work/gr-fe/bryan/data/YEAST/02_processed

[EXPR] Saved (2417, 103) to /work/gr-fe/bryan/data/YEAST/02_processed/EXPR.pkl


,0,1,2,3,4,5,6,7,8,9,...,93,94,95,96,97,98,99,100,101,102
ID,,,,,,,,,,,,,,,,,,,,,
000UH,0.152205,0.033165,0.174814,0.061704,0.148062,0.064145,0.147432,0.058457,0.149786,0.114706,...,-0.056918,0.054003,0.051133,0.050457,0.046689,-0.053047,-0.063318,-0.062440,-0.030184,0.006750
02KOH,-0.030334,0.251017,0.103362,-0.038219,-0.100961,-0.030429,0.009821,-0.033011,0.010851,0.025187,...,-0.130644,0.017761,0.142255,0.138558,0.168424,-0.113521,0.150490,0.318243,-0.106129,-0.174929
03P0J,-0.035372,-0.006865,-0.020484,0.063093,-0.075701,0.118055,-0.134383,0.060565,-0.018069,0.074334,...,-0.029846,-0.045711,-0.042562,-0.046287,-0.046498,-0.024897,0.228166,-0.051131,0.022426,0.047288
05AIF,0.015089,0.073954,0.023296,-0.093508,0.025380,-0.062028,0.040267,-0.060581,0.032808,0.095193,...,-0.115480,-0.004234,0.122726,0.125482,0.110430,-0.104427,0.059652,-0.118384,-0.106356,0.040329
06D78,-0.002940,-0.043663,-0.039564,-0.196506,-0.133996,-0.027484,-0.033110,0.220659,-0.131564,0.071299,...,0.104165,-0.034423,-0.032029,0.136194,-0.035144,0.439551,-0.044345,-0.003927,-0.008246,-0.126706
